# Local ONNX 추론 테스트 — Kaggle 결과와 차이 진단용

> **목적**: Kaggle 에서 `test_018=4` 로 예측됐는데 Local 에서는 `0` 으로 나오는지 확인.
>
> - 가속 없음 (단순 sequential ONNX forward)
> - 같은 5개 artifact 사용 (`kaggle_artifacts/`)
> - Local test 폴더 사용
> - test_018 의 raw softmax 상세 출력
> - submission.csv → `local_submission.csv` 로 별도 저장 (Kaggle 것과 md5 비교용)

## [Cell 1] Imports + 환경 정보

In [ ]:
import os
import time
import json
import pickle
import hashlib
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import onnxruntime as ort
import sklearn
import skimage
from skimage.feature import local_binary_pattern

print("━" * 60)
print("  🖥️  Local Environment")
print("━" * 60)
print(f"  OS                  : {platform.platform()}")
print(f"  Python              : {platform.python_version()}")
print(f"  CPU arch            : {platform.machine()}")
print(f"  numpy               : {np.__version__}")
print(f"  pandas              : {pd.__version__}")
print(f"  scikit-learn        : {sklearn.__version__}")
print(f"  scikit-image        : {skimage.__version__}")
print(f"  onnxruntime         : {ort.__version__}")
print(f"  opencv              : {cv2.__version__}")
print("━" * 60)

## [Cell 2] 경로 + Test 폴더 진단

In [ ]:
# === 경로 정의 ===
try:
    _HERE = Path(__file__).resolve().parent
except NameError:
    _vsc = globals().get('__vsc_ipynb_file__')
    _HERE = Path(_vsc).resolve().parent if _vsc else Path.cwd().resolve()

ARTIFACTS_DIR = _HERE / 'kaggle_artifacts'

# Project root 자동 탐색
def find_project_root(start):
    for p in [start, *start.parents]:
        if (p / 'competition_dataset' / 'NEU-DET_open').is_dir():
            return p
    return None

PROJECT_ROOT = find_project_root(_HERE)
assert PROJECT_ROOT is not None, "competition_dataset 못 찾음"

TEST_DIR = PROJECT_ROOT / 'competition_dataset' / 'NEU-DET_open' / 'test' / 'images'
OUT_DIR = _HERE / 'local_test_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 _HERE         : {_HERE}")
print(f"📂 ARTIFACTS_DIR : {ARTIFACTS_DIR}  ({'✅' if ARTIFACTS_DIR.exists() else '❌'})")
print(f"📂 PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"📂 TEST_DIR      : {TEST_DIR}  ({'✅' if TEST_DIR.exists() else '❌'})")
print(f"📂 OUT_DIR       : {OUT_DIR}")

# === Test 폴더 구조 진단 ===
print(f"\n📁 TEST_DIR 구조 진단:")
if TEST_DIR.exists():
    items = sorted(TEST_DIR.iterdir())
    direct_jpgs = [f for f in items if f.is_file() and f.suffix.lower() in ('.jpg', '.png')]
    subdirs = [d for d in items if d.is_dir()]
    print(f"  직접 .jpg/.png: {len(direct_jpgs)}장")
    if direct_jpgs[:5]:
        print(f"     샘플: {[f.name for f in direct_jpgs[:5]]}")
    print(f"  하위 폴더: {len(subdirs)}개 {[d.name for d in subdirs]}")

    # test_018.jpg 존재 + MD5
    test_018 = TEST_DIR / 'test_018.jpg'
    if test_018.exists():
        md5 = hashlib.md5(test_018.read_bytes()).hexdigest()
        print(f"\n🎯 test_018.jpg 발견")
        print(f"   path : {test_018}")
        print(f"   size : {test_018.stat().st_size:,} bytes")
        print(f"   md5  : {md5}")
    else:
        print(f"\n⚠️ test_018.jpg 직접 안 보임 — 하위 폴더 검색:")
        from itertools import chain
        for p in chain(*(d.glob('test_018.jpg') for d in subdirs)):
            md5 = hashlib.md5(p.read_bytes()).hexdigest()
            print(f"   path : {p}")
            print(f"   size : {p.stat().st_size:,} bytes")
            print(f"   md5  : {md5}")
else:
    print(f"  ❌ TEST_DIR 없음")

## [Cell 3] ONNX session + thresholds + AH12 load

In [ ]:
# Session options — 단순 sequential
sess_opts = ort.SessionOptions()
sess_opts.intra_op_num_threads = max(1, os.cpu_count() or 4)
sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

# FP32 ONNX (INT8 안 씀 — 환경 차이 영향 최소화)
sess_beta  = ort.InferenceSession(str(ARTIFACTS_DIR / 'student_BETA-LION.onnx'),
                                   sess_options=sess_opts, providers=['CPUExecutionProvider'])
sess_bidir = ort.InferenceSession(str(ARTIFACTS_DIR / 'student_BIDIR.onnx'),
                                   sess_options=sess_opts, providers=['CPUExecutionProvider'])
sess_cnxt  = ort.InferenceSession(str(ARTIFACTS_DIR / 'teacher_convnext_tiny.onnx'),
                                   sess_options=sess_opts, providers=['CPUExecutionProvider'])

INAME_BETA  = sess_beta.get_inputs()[0].name
INAME_BIDIR = sess_bidir.get_inputs()[0].name
INAME_CNXT  = sess_cnxt.get_inputs()[0].name

# MD5 (Kaggle 의 것과 비교 가능)
md5_beta  = hashlib.md5((ARTIFACTS_DIR / 'student_BETA-LION.onnx').read_bytes()).hexdigest()
md5_bidir = hashlib.md5((ARTIFACTS_DIR / 'student_BIDIR.onnx').read_bytes()).hexdigest()
md5_cnxt  = hashlib.md5((ARTIFACTS_DIR / 'teacher_convnext_tiny.onnx').read_bytes()).hexdigest()

print(f"✅ Loaded 3 ONNX sessions (FP32, sequential)")
print(f"  student_BETA-LION       md5={md5_beta}")
print(f"  student_BIDIR           md5={md5_bidir}")
print(f"  teacher_convnext_tiny   md5={md5_cnxt}")

# Thresholds + AH12 sklearn
with open(ARTIFACTS_DIR / 'thresholds.json') as f:
    TH = json.load(f)
with open(ARTIFACTS_DIR / 'ah12_state.pkl', 'rb') as f:
    AH12 = pickle.load(f)

# Constants
class_names = TH['class_names']
CRA, INC, PAT, PIT, ROL, SCR = 0, 1, 2, 3, 4, 5
IMG_SIZE = int(TH['IMG_SIZE'])
NUM_CLASSES = int(TH['NUM_CLASSES'])
wA = float(TH['wA'])
class_bias = np.array(TH['class_bias'], dtype=np.float32)
ALLOWED_TARGETS = [(int(t[0]), float(t[1])) for t in TH['ALLOWED_TARGETS']]
ALLOWED_MAP = {t: cf for (t, cf) in ALLOWED_TARGETS}
P_VETO = float(TH['P_VETO'])
PIT_CNXT_FLOOR = float(TH['R4_cnxt_floor'])
PIT_BETA_FLOOR = float(TH['R4_beta_pit_floor'])
PIT_PFLOOR = float(TH['R4_p_pit_floor'])
BIDIR_ROL_VETO = float(TH['BIDIR_ROL_VETO'])
RULE_3WAY_VALID = bool(TH['rule_3way_valid'])
R4_VALID = bool(TH['r4_valid'])
AH12_VALID = bool(TH['ah12_valid'])

print(f"\n✅ Loaded thresholds + AH12 state")
print(f"  wA={wA}, class_bias={class_bias.tolist()}")

## [Cell 4] Test 이미지 preload (cv2)

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)

# Test files (sorted)
test_files = sorted([f for f in os.listdir(TEST_DIR)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
N = len(test_files)
print(f"Test images (직접 폴더): {N}")
if N == 0:
    raise RuntimeError("TEST_DIR 직접 안에 .jpg 없음 — class 별 하위 폴더 구조인지 확인 필요")

# ⭐ V9-HONEST .pth 학습/추론과 동일 — PIL.Image.open + PIL.BILINEAR resize
#   (이전 cv2 경로는 JPEG 디코더 + bilinear kernel 이 미세하게 달라 boundary
#    sample 의 argmax 가 뒤집힘. test_018 = 4 → 0 회복용)
def load_and_preprocess(fn):
    # 🔒 worker process (fork/spawn 모두) 에서도 안전하게 — 함수 내부 import
    from PIL import Image
    pil = Image.open(TEST_DIR / fn).convert('RGB').resize(
        (IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    img_u8 = np.array(pil)  # uint8 RGB, PIL-resized (compute_feat13 source)
    arr = img_u8.astype(np.float32).transpose(2, 0, 1) / 255.0
    arr = (arr - MEAN) / STD
    return arr.astype(np.float32), img_u8

print("Preloading ...")
results = [load_and_preprocess(fn) for fn in test_files]
X = np.stack([r[0] for r in results])
resized_imgs = [r[1] for r in results]
print(f"X shape: {X.shape}")

## [Cell 5] 13D feature + p_pit

In [ ]:
LBP_P, LBP_R, LBP_BINS = 8, 1, 10

def fft_power_ratio(gray_f32):
    F_ = np.fft.fftshift(np.fft.fft2(gray_f32)); mag = np.abs(F_)
    H, W = gray_f32.shape; cy, cx = H // 2, W // 2
    yy, xx = np.indices((H, W))
    r = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    r_n = r / r.max()
    return float(mag[(r_n >= 0.4) & (r_n <= 1.0)].sum() / (mag.sum() + 1e-8))

def coherence(gray_f32):
    gx = cv2.Sobel(gray_f32, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray_f32, cv2.CV_32F, 0, 1, ksize=3)
    Sxx = cv2.GaussianBlur(gx * gx, (5, 5), 1.0)
    Sxy = cv2.GaussianBlur(gx * gy, (5, 5), 1.0)
    Syy = cv2.GaussianBlur(gy * gy, (5, 5), 1.0)
    tr = Sxx + Syy
    det = Sxx * Syy - Sxy * Sxy
    sq = np.sqrt(np.maximum(tr * tr / 4 - det, 0))
    return float(((tr / 2 + sq - (tr / 2 - sq)) /
                  (tr / 2 + sq + tr / 2 - sq + 1e-8)).mean())

def cc_count(gray_u8):
    _, th = cv2.threshold(gray_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    num, _ = cv2.connectedComponents(th)
    return int(num)

def lbp_hist(gray_u8):
    lbp = local_binary_pattern(gray_u8, LBP_P, LBP_R, method='uniform')
    h, _ = np.histogram(lbp.ravel(), bins=LBP_BINS, range=(0, LBP_BINS), density=True)
    return h.astype(np.float32)

def compute_feat13(rgb_u8):
    gray_u8 = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2GRAY)
    gray_f32 = gray_u8.astype(np.float32) / 255.0
    return np.concatenate([
        [fft_power_ratio(gray_f32),
         coherence(gray_f32),
         float(cc_count(gray_u8))],
        lbp_hist(gray_u8),
    ])

print("13D feature 추출...")
test_feats = np.stack([compute_feat13(img) for img in resized_imgs])
test_p_pit = AH12['clf'].predict_proba(AH12['scaler'].transform(test_feats))[:, 1]
print(f"test_feats shape: {test_feats.shape}")

## [Cell 6] ONNX forward (단순 sequential — 가속 없음)

In [ ]:
def softmax_np(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

print("ONNX forward (sequential, full 180 for all 3 models)...")
t0 = time.perf_counter()
lA = sess_beta.run(None,  {INAME_BETA:  X})[0]
lB = sess_bidir.run(None, {INAME_BIDIR: X})[0]
lC = sess_cnxt.run(None,  {INAME_CNXT:  X})[0]
t_fwd = time.perf_counter() - t0
print(f"Forward time: {t_fwd:.3f}s (참고용, 가속 안 함)")

probA = softmax_np(lA); probB = softmax_np(lB); probC = softmax_np(lC)

# Champion
champ_prob  = wA * probA + (1 - wA) * probB
champ_final = softmax_np(np.log(champ_prob + 1e-12) + class_bias[None, :])
champ_pred  = champ_final.argmax(1)
champ_conf  = champ_final.max(1)

beta_argmax  = probA.argmax(1); beta_conf  = probA.max(1)
bidir_argmax = probB.argmax(1); bidir_conf = probB.max(1)
cnxt_argmax  = probC.argmax(1); cnxt_conf  = probC.max(1)

# ⭐ test_018 상세 — Kaggle 과 비교용
if 'test_018.jpg' in test_files:
    idx = test_files.index('test_018.jpg')
    print(f"\n" + "=" * 70)
    print(f"🎯 test_018.jpg — Local 에서의 결과 (Kaggle 과 비교)")
    print("=" * 70)
    print(f"  ⭐ Kaggle 결과 : 4 (rolled-in_scale)")
    print(f"  ⭐ Local 결과  : {champ_pred[idx]} ({class_names[champ_pred[idx]]})")
    print(f"  ⭐ 사용자 정답 : 0 (crazing)")
    print()
    print(f"  BETA  prob : {probA[idx].round(4).tolist()}  → argmax={beta_argmax[idx]} ({class_names[beta_argmax[idx]]})")
    print(f"  BIDIR prob : {probB[idx].round(4).tolist()}  → argmax={bidir_argmax[idx]} ({class_names[bidir_argmax[idx]]})")
    print(f"  CnxT  prob : {probC[idx].round(4).tolist()}  → argmax={cnxt_argmax[idx]} ({class_names[cnxt_argmax[idx]]})")
    print(f"  Champion   : {champ_final[idx].round(4).tolist()}  → argmax={champ_pred[idx]} ({class_names[champ_pred[idx]]})")
    print(f"  p_pit      : {test_p_pit[idx]:.4f}")
    print("=" * 70)

## [Cell 7] Override stack 적용

In [ ]:
final_pred = champ_pred.copy()
fired_log = []
veto_log = []

# R3
if RULE_3WAY_VALID and ALLOWED_MAP:
    for i in range(N):
        if not (champ_pred[i] != beta_argmax[i]
                and beta_argmax[i] == cnxt_argmax[i]
                and int(beta_argmax[i]) in ALLOWED_MAP
                and cnxt_conf[i] >= ALLOWED_MAP[int(beta_argmax[i])]):
            continue
        from_cls = int(champ_pred[i]); to_cls = int(beta_argmax[i])
        if from_cls == PIT and to_cls == INC and float(test_p_pit[i]) >= P_VETO:
            veto_log.append({'id': test_files[i], 'rule': 'R3_pit_to_inc_veto',
                              'p_pit': float(test_p_pit[i])})
            continue
        final_pred[i] = to_cls
        fired_log.append({'id': test_files[i], 'rule': 'R3',
                           'from': from_cls, 'to': to_cls,
                           'p_pit': float(test_p_pit[i])})

# R4
if R4_VALID:
    beta_pit  = probA[:, PIT]
    cnxt_pit  = probC[:, PIT]
    bidir_rol = probB[:, ROL]
    fired_ids = {ov['id'] for ov in fired_log}
    for i in range(N):
        if test_files[i] in fired_ids: continue
        if not (champ_pred[i] != beta_argmax[i]
                and beta_argmax[i] == cnxt_argmax[i]
                and beta_argmax[i] == PIT
                and cnxt_pit[i]  >= PIT_CNXT_FLOOR
                and beta_pit[i]  >= PIT_BETA_FLOOR
                and float(test_p_pit[i]) >= PIT_PFLOOR):
            continue
        if bidir_rol[i] >= BIDIR_ROL_VETO:
            veto_log.append({'id': test_files[i], 'rule': 'R4_bidir_rol_veto',
                              'bidir_rol': float(bidir_rol[i])})
            continue
        final_pred[i] = PIT
        fired_log.append({'id': test_files[i], 'rule': 'R4',
                           'from': int(champ_pred[i]), 'to': PIT,
                           'beta_pit': float(beta_pit[i]),
                           'cnxt_pit': float(cnxt_pit[i]),
                           'p_pit': float(test_p_pit[i])})

# AH12
if AH12_VALID:
    mu_inc3, Sinv_inc3, ld_inc3 = AH12['mu_inc3'], AH12['Sinv_inc3'], AH12['ld_inc3']
    mu_pit3, Sinv_pit3, ld_pit3 = AH12['mu_pit3'], AH12['Sinv_pit3'], AH12['ld_pit3']
    A_r1_fft = AH12['A_r1_fft']; A_r1_coh = AH12['A_r1_coh']
    A_r2_thr = AH12['A_r2_thr']; A_r3_thr = AH12['A_r3_thr']; A_r4_thr = AH12['A_r4_thr']
    def _ll(x, mu, Sinv, ld): return -0.5 * ((x - mu) @ Sinv @ (x - mu)) - 0.5 * ld
    def _dmaha(x, mu, Sinv): return float((x - mu) @ Sinv @ (x - mu))
    fired_ids = {ov['id'] for ov in fired_log}
    for i in range(N):
        if test_files[i] in fired_ids: continue
        if int(final_pred[i]) != INC: continue
        x = test_feats[i]
        r1 = (x[0] < A_r1_fft) and (x[1] > A_r1_coh)
        if not r1: continue
        r2 = (_ll(x[:3], mu_pit3, Sinv_pit3, ld_pit3) -
              _ll(x[:3], mu_inc3, Sinv_inc3, ld_inc3)) > A_r2_thr
        if not r2: continue
        dp = _dmaha(x[:3], mu_pit3, Sinv_pit3)
        di = _dmaha(x[:3], mu_inc3, Sinv_inc3)
        r3 = (dp < di) and (dp < A_r3_thr)
        if not r3: continue
        r4 = float(test_p_pit[i]) > A_r4_thr
        if not r4: continue
        final_pred[i] = PIT
        fired_log.append({'id': test_files[i], 'rule': 'AH12',
                           'from': INC, 'to': PIT,
                           'p_pit': float(test_p_pit[i])})

print(f"Override fires: {len(fired_log)}")
for ov in fired_log:
    extra = ', '.join(f"{k}={v:.3f}" if isinstance(v, float) else f"{k}={v}"
                       for k, v in ov.items() if k not in ('id', 'rule', 'from', 'to'))
    print(f"  [{ov['rule']:5s}] {ov['id']}: {class_names[ov['from']]} → {class_names[ov['to']]}  {extra}")

print(f"\nVeto blocks: {len(veto_log)}")
for v in veto_log:
    extra = ', '.join(f"{k}={v_:.3f}" if isinstance(v_, float) else f"{k}={v_}"
                       for k, v_ in v.items() if k not in ('id', 'rule'))
    print(f"  [{v['rule']}] {v['id']}: {extra}")

print(f"\nFinal prediction distribution:")
for c, name in enumerate(class_names):
    print(f"  {name:20s}: {int((final_pred == c).sum())}")

## [Cell 8] Local submission.csv 저장 + Kaggle 결과와 sample-by-sample 비교

In [ ]:
# Local submission 저장
submission = pd.DataFrame({
    'Id': test_files,
    'Expected': final_pred.astype(int),
    'inference_time_sec': 0.0,   # local 이라 측정 안 함
})
local_sub_path = OUT_DIR / 'local_submission.csv'
submission.to_csv(local_sub_path, index=False)
local_md5 = hashlib.md5(local_sub_path.read_bytes()).hexdigest()

print("━" * 70)
print(f"  local_submission.csv saved → {local_sub_path}")
print(f"  md5  : {local_md5}")
print(f"  rows : {len(submission)}")
print("━" * 70)

# ⭐ test_018 최종 결과
if 'test_018.jpg' in test_files:
    idx = test_files.index('test_018.jpg')
    print(f"\n🎯 test_018.jpg 최종 결과 비교")
    print(f"  사용자 정답  : 0 (crazing)")
    print(f"  Kaggle 결과 : 4 (rolled-in_scale)")
    print(f"  Local 결과  : {final_pred[idx]} ({class_names[final_pred[idx]]})")
    print()
    if final_pred[idx] == 0:
        print("  → Local 에서는 정답 ✅. Kaggle 환경 차이 (cv2/libjpeg/BLAS) 가 원인!")
    elif final_pred[idx] == 4:
        print("  → Local 도 wrong. ONNX export 자체의 정확도 문제일 수 있음.")
    else:
        print(f"  → 또 다른 결과. 모델/이미지 둘 다 의심.")

# ⭐ V9-HONEST 의 원본 submission 과 비교
v9_sub_path = PROJECT_ROOT / 'smart-factory_obisian' / 'output' / 'decisions' / 'V9-HONEST-179-180-FROM-SCRATCH' / 'submission.csv'
if v9_sub_path.exists():
    v9_df = pd.read_csv(v9_sub_path)
    v9_md5 = hashlib.md5(v9_sub_path.read_bytes()).hexdigest()

    # 정렬 후 비교
    v9_sorted = v9_df.sort_values('Id').reset_index(drop=True)
    local_sorted = submission.sort_values('Id').reset_index(drop=True)

    diff_mask = v9_sorted['Expected'] != local_sorted['Expected']
    n_diff = int(diff_mask.sum())

    print(f"\n📊 V9-HONEST 원본 vs Local ONNX 비교")
    print(f"  V9 submission md5   : {v9_md5}")
    print(f"  Local submission md5: {local_md5}")
    print(f"  일치 sample         : {len(v9_sorted) - n_diff}/{len(v9_sorted)}")

    if n_diff > 0:
        print(f"\n  다른 sample ({n_diff}개):")
        for _, row in v9_sorted[diff_mask].iterrows():
            local_pred = local_sorted[local_sorted['Id'] == row['Id']]['Expected'].iloc[0]
            print(f"    {row['Id']}: V9 PyTorch={row['Expected']} ({class_names[row['Expected']]}) "
                  f"vs Local ONNX={local_pred} ({class_names[local_pred]})")
    else:
        print(f"  ✅ V9 PyTorch (.pth) 와 Local ONNX 가 bit-exact 일치")
else:
    print(f"\n⚠️ V9-HONEST submission.csv 없음 ({v9_sub_path})")

## [Cell 9] 진단 종합

### 결과 해석

| Local ONNX 의 test_018 | 진단 |
|---|---|
| **0 (crazing)** | Local + Kaggle 환경 차이가 원인. ONNX 변환은 OK. Kaggle 의 cv2 libjpeg 또는 BLAS 가 의심 |
| **4 (rolled-in_scale)** | ONNX 변환 자체에 정확도 손실. PyTorch (.pth) 결과와 다름. 다른 opset 또는 PyTorch 직접 사용 필요 |
| **다른 값** | 더 큰 문제 — 모델 / 이미지 / 코드 어딘가에 차이 |

### V9 PyTorch vs Local ONNX 비교

만약 **bit-exact 일치** → ONNX 변환 완벽. Kaggle 결과 차이는 환경 문제.
만약 **차이 발생** → ONNX export 시 양자화 손실 발생. 그 sample 들이 boundary case.